#### Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate pillow


In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
from PIL import Image
from IPython.display import display

DISPLAY_MAX_SIZE = 300


def _show(img_or_path):
    """Display an image at a readable notebook size without changing the original."""
    img = Image.open(img_or_path) if isinstance(img_or_path, str) else img_or_path
    preview = img.copy()
    preview.thumbnail((DISPLAY_MAX_SIZE, DISPLAY_MAX_SIZE))
    display(preview)


#### Dataset 

In [ ]:
###  Do not need to rerun this unless the runtime restarted. ###

from datasets import load_dataset
from itertools import islice

DATASET_ID = "Supermaxman/esa-hubble"

# Load dataset as streaming dataset.
dataset_stream = load_dataset(DATASET_ID, split="train", streaming=True)

# Choose how many rows/images you want to inspect.
N = 266

rows = list(islice(dataset_stream, N))

print("Loaded rows:", len(rows))
print("Available columns:")
print(rows[0].keys())

Resolving data files:   0%|          | 0/200 [00:00<?, ?it/s]

Loaded rows: 266
Available columns:
dict_keys(['image', 'text', 'id', 'title', 'description', 'credits', 'url', 'Id', 'Type', 'Release date', 'Related releases', 'Size', 'Name', 'Distance', 'Constellation', 'Category', 'Position (RA)', 'Position (Dec)', 'Field of view', 'Orientation', 'width', 'height', 'file_size', 'crop_w', 'crop_h', 'cropped', 'Related science announcements', 'Related announcements'])


#### Project Reference Files

The AVM reference text and true AVM labels are loaded from the project GitHub repo, so they do not need to be embedded in this notebook.


In [ ]:
import urllib.request

PROJECT_RAW_BASE_URL = "https://raw.githubusercontent.com/QihanQG/ECS-189G-final-project/main"
AVM_REFERENCE_URL = f"{PROJECT_RAW_BASE_URL}/AVM_Reference"
TRUE_AVM_CODES_URL = f"{PROJECT_RAW_BASE_URL}/TRUE_AVM_CODES"


def load_text_from_url(url):
    """Load a UTF-8 text file from a public URL."""
    with urllib.request.urlopen(url, timeout=30) as response:
        return response.read().decode("utf-8")


def parse_true_avm_codes(text):
    """Parse lines like `12: C.5.1.1, C.5.4.6` into a list of code lists."""
    labels_by_row = {}

    for line in text.splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue

        row_text, codes_text = line.split(":", 1)
        row_index = int(row_text.strip())
        codes = [code.strip() for code in codes_text.split(",") if code.strip()]
        labels_by_row[row_index] = codes

    expected_indices = list(range(len(labels_by_row)))
    actual_indices = sorted(labels_by_row)
    if actual_indices != expected_indices:
        raise ValueError("TRUE_AVM_CODES rows are missing or out of order.")

    return [labels_by_row[i] for i in expected_indices]


avm_reference_text = load_text_from_url(AVM_REFERENCE_URL)
TRUE_AVM_CODES = parse_true_avm_codes(load_text_from_url(TRUE_AVM_CODES_URL))


# Example 
print(f"Loaded AVM reference text: {len(avm_reference_text):,} characters")
print(f"Loaded AVM ground-truth labels: {len(TRUE_AVM_CODES)} rows")
print("Row 0 true AVM code(s):", TRUE_AVM_CODES[0])


Loaded AVM reference text: 7,778 characters
Loaded AVM ground-truth labels: 266 rows
Row 0 true AVM code(s): ['B.3.6.4.1', 'B.4.1.3']


#### Chat

# Chat with the actual dataset

In [20]:
def ask_qwen_restricted(image, prompt, system_prompt="", max_new_tokens=256):
    """
    Ask Qwen a question with a domain-restricting system prompt.
    """
    if isinstance(image, str):
        img = Image.open(image).convert("RGB")
    else:
        img = image.convert("RGB") if isinstance(image, Image.Image) else Image.fromarray(image).convert("RGB")

    _show(img)

    # Initialize messages list
    messages = []

    # Add system prompt if provided to restrict the domain
    if system_prompt:
        messages.append({
            "role": "system",
            "content": [{"type": "text", "text": system_prompt}]
        })

    # Add the user prompt with the image
    messages.append({
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ],
    })

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    reply = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    print(reply)
    return reply

#### The Prompt:

In [21]:
avm_task_instructions = """
You are an expert astronomical image classifier. Given an astronomical image, your task is to identify the type(s) of object(s) or phenomenon(a) depicted and express each as an AVM 1.1 code.

AVM 1.1 Code Format:
Each code has the form <Scale>.<Taxonomy>, where:

Scale (single letter prefix):
A = Solar System
B = Milky Way
C = Local Universe
D = Early Universe
E = Unspecified
Taxonomy (dot-separated numbers following the scale letter), e.g.:
B.4.1.3 â†’ Milky Way : Nebula : Type : Planetary
C.5.1.1 â†’ Local Universe : Galaxy : Type : Spiral
D.5.5.3 â†’ Early Universe : Galaxy : Grouping : Cluster
Instructions:

Carefully examine the image and identify all astronomical object types or phenomena present.
For each identified type, select the most specific matching AVM 1.1 code.
An image may contain more than one type â€” if so, provide one code per type, separated by commas.
Output only the AVM code(s), nothing else. No explanations, no labels, no extra text.
Output format:

Single type: B.4.1.3
Multiple types: C.5.1.1, C.5.1.2
"""

avm_system_prompt = avm_reference_text + "\n\n" + avm_task_instructions


#### Batch Prediction Output
 
 Run Qwen on every loaded dataset image and save predictions as `index: AVM_CODE, AVM_CODE` lines.


In [ ]:
import re
from pathlib import Path

# Change this if you want to use different prompt for qwen prediction so that it doesn't overwrite the existing saved files.
PROMPT_RUN_NAME = "strict_codes_v1"
PREDICTIONS_OUTPUT_PATH = Path(f"qwen_predicted_avm_codes_{PROMPT_RUN_NAME}.txt")
AVM_CODE_PATTERN = re.compile(r"\b[A-EX]\.\d+(?:\.\d+)*\b")


def ask_qwen_avm_codes_no_display(image, system_prompt, max_new_tokens=100):
    """Ask Qwen for AVM code predictions without displaying the image."""
    if isinstance(image, str):
        img = Image.open(image).convert("RGB")
    else:
        img = image.convert("RGB") if isinstance(image, Image.Image) else Image.fromarray(image).convert("RGB")

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompt}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": "Return only the AVM code or codes for this image."},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    reply = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    return reply.strip()


def extract_avm_codes(model_reply):
    """Extract AVM-looking codes from Qwen's reply."""
    codes = AVM_CODE_PATTERN.findall(model_reply)
    return codes if codes else [model_reply.replace("\n", " ").strip()]


def save_prediction_lines(lines, output_path):
    """Save prediction lines."""
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


prediction_lines = []

for row_index, row in enumerate(rows):
    print(f"Processing {row_index + 1}/{len(rows)}")

    raw_prediction = ask_qwen_avm_codes_no_display(
        row["image"],
        system_prompt=avm_system_prompt,
        max_new_tokens=100,
    )
    predicted_codes = extract_avm_codes(raw_prediction)
    prediction_lines.append(f"{row_index}: {', '.join(predicted_codes)}")

    # Save after each image so progress is not lost if the runtime stops.
    save_prediction_lines(prediction_lines, PREDICTIONS_OUTPUT_PATH)

print(f"Saved {len(prediction_lines)} predictions to {PREDICTIONS_OUTPUT_PATH}")

Processing 1/266
Processing 2/266
Processing 3/266
Processing 4/266
Processing 5/266
Processing 6/266
Processing 7/266
Processing 8/266
Processing 9/266
Processing 10/266
Processing 11/266
Processing 12/266
Processing 13/266
Processing 14/266
Processing 15/266
Processing 16/266
Processing 17/266
Processing 18/266
Processing 19/266
Processing 20/266
Processing 21/266
Processing 22/266
Processing 23/266
Processing 24/266
Processing 25/266
Processing 26/266
Processing 27/266
Processing 28/266
Processing 29/266
Processing 30/266
Processing 31/266
Processing 32/266
Processing 33/266
Processing 34/266
Processing 35/266
Processing 36/266
Processing 37/266
Processing 38/266
Processing 39/266
Processing 40/266
Processing 41/266
Processing 42/266
Processing 43/266
Processing 44/266
Processing 45/266
Processing 46/266
Processing 47/266
Processing 48/266
Processing 49/266
Processing 50/266
Processing 51/266
Processing 52/266
Processing 53/266
Processing 54/266
Processing 55/266
Processing 56/266
P

#### Qwen output/prediction 

In [29]:
from pathlib import Path


#output the whole thing
print(PREDICTIONS_OUTPUT_PATH.resolve())
print(PREDICTIONS_OUTPUT_PATH.read_text(encoding="utf-8"))#output the whole thing

/content/qwen_predicted_avm_codes.txt
0: C.4.1.4
1: C.5.1.7
2: B.4.1.2
3: B.3.5.1.2
4: B.4.2.3.1
5: C.5.3.2.1
6: C.5.1.1
7: C.5.5.3
8: C.5.1.7, C.5.5.3
9: C.5.1.1
10: B.3.5.1.2, B.4.1.2
11: C.5.1.7
12: C.5.1.1
13: C.5.1.1
14: C.5.5.3
15: C.5.1.7
16: C.5.5.4
17: C.4.1.4
18: B.4.1.4
19: C.4.1.4
20: C.5.1.7, C.5.5.1
21: C.5.1.7, C.5.5.1
22: C.5.1.7
23: C.5.1.7
24: C.5.1.7
25: C.5.1.1
26: C.5.1.1
27: C.5.1.1
28: C.5.1.7
29: C.5.1.1
30: C.5.1.7, C.5.1.8
31: C.5.1.1
32: C.5.1.7
33: B.4.1.2
34: C.4.2.3.1
35: C.5.1.1
36: C.5.1.7
37: C.5.1.1
38: C.5.1.1
39: C.5.1.1
40: C.5.1.1
41: C.5.1.7, C.5.5.1
42: C.5.1.7
43: C.5.1.1
44: B.3.6.4.1
45: C.5.5.3
46: B.5.2.1, B.7.1.1
47: B.4.1.2
48: B.4.1.2, B.4.2.1.1
49: C.4.1.4
50: B.3.1.3
51: C.5.1.7
52: B.3.1.3, B.3.5.1.2
53: C.5.5.4
54: C.5.1.7
55: C.5.5.4
56: C.4.1.2
57: C.3.1.8, C.3.1.9.1, C.3.7.2.1
58: B.3.6.4.1
59: C.5.1.7
60: C.5.1.7
61: C.5.1.7
62: C.4.1.2, C.3.1.3
63: C.4.1.2
64: C.5.1.7
65: C.4.1.2
66: C.4.1.2, C.3.5.1.2
67: C.5.1.7
68: C.4.1.2
69: